# Case 4: Caspase Inactive & Viability Inactive

Analysis of compounds where:
- **Caspase = Inactive** (has at least one Inactive outcome AND no Active outcomes)
- **Viability = Inactive** (has at least one Inactive outcome AND no Active outcomes)

**Classification Method:** STRICT
- Active takes precedence: if ANY test shows Active, the CID is classified as Active
- Inactive requires explicit Inactive outcome (not just absence of Active)
- CIDs with only Inconclusive outcomes are excluded

**Biological Interpretation:** Non-toxic compounds

**Assay Types:**
- `caspase` - Caspase-3/7 assay (14 cell lines)
- `viability` - Viability assay (CHO-K1)
- `mitochondrial_viability_hepg2` - Mitochondrial viability (HepG2)
- `caspase_viability_hepg2` - Caspase viability counter-screen (HepG2)

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)
pd.set_option('display.max_rows', 100)

# Load the merged dataset
df = pd.read_csv('output/merged_caspase_viability_final.csv')

print(f"Dataset Shape: {df.shape}")
print(f"Unique CIDs: {df['CID'].nunique():,}")
print(f"\nAssay Types: {df['assay_type'].unique().tolist()}")
print(f"\nCell Lines ({df['cell_line'].nunique()}): {df['cell_line'].unique().tolist()}")

Dataset Shape: (26939, 12)
Unique CIDs: 8,777

Assay Types: ['viability', 'caspase', 'mitochondrial_viability_hepg2', 'caspase_viability_hepg2']

Cell Lines (14): ['CHO-K1', 'HepG2', 'SHSY5Y', 'Hek293', 'HUV-EC-C', 'Jurkat', 'H-4-II-E', 'N2a', 'Mesangial', 'NIH 3T3', 'SK-N-SH', 'BJ', 'MRC-5', 'Renal_Proximal_Tubule']


In [2]:
# Define viability types
VIABILITY_TYPES = ['viability', 'mitochondrial_viability_hepg2', 'caspase_viability_hepg2']

print("Viability assay types used in this analysis:")
print(f"  - viability (CHO-K1)")
print(f"  - mitochondrial_viability_hepg2 (HepG2)")
print(f"  - caspase_viability_hepg2 (HepG2)")

Viability assay types used in this analysis:
  - viability (CHO-K1)
  - mitochondrial_viability_hepg2 (HepG2)
  - caspase_viability_hepg2 (HepG2)


---
## Block 1: Overview - Case 4 Criteria (STRICT Method)

Find CIDs where:
- Caspase = **Inactive** (has at least one Inactive outcome AND no Active outcomes)
- Viability = **Inactive** (has at least one Inactive outcome AND no Active outcomes)

**Note:** Only CIDs tested in BOTH assay types are included.

In [3]:
# STRICT METHOD: 
# - Active takes precedence
# - Inactive = has Inactive outcome AND no Active outcome
# Case 4 = Caspase Truly Inactive AND Viability Truly Inactive

# Get all CIDs tested in caspase assays
caspase_df = df[df['assay_type'] == 'caspase']
caspase_all_cids = set(caspase_df['CID'].unique())
caspase_active_cids = set(caspase_df[caspase_df['outcome'] == 'Active']['CID'].unique())
caspase_inactive_cids = set(caspase_df[caspase_df['outcome'] == 'Inactive']['CID'].unique())

# Get all CIDs tested in viability assays
viability_df = df[df['assay_type'].isin(VIABILITY_TYPES)]
viability_all_cids = set(viability_df['CID'].unique())
viability_active_cids = set(viability_df[viability_df['outcome'] == 'Active']['CID'].unique())
viability_inactive_cids = set(viability_df[viability_df['outcome'] == 'Inactive']['CID'].unique())

# Common CIDs - tested in BOTH assay types
common_cids = caspase_all_cids.intersection(viability_all_cids)

# Caspase Truly Inactive = has Inactive outcome but NO Active outcome (in common CIDs)
caspase_truly_inactive = caspase_inactive_cids.intersection(common_cids) - caspase_active_cids

# Viability Truly Inactive = has Inactive outcome but NO Active outcome (in common CIDs)
viability_truly_inactive = viability_inactive_cids.intersection(common_cids) - viability_active_cids

print("STRICT METHOD: Active takes precedence, Inactive = has Inactive & no Active")
print("="*70)
print(f"CIDs tested in Caspase assays:     {len(caspase_all_cids):,}")
print(f"CIDs tested in Viability assays:   {len(viability_all_cids):,}")
print(f"CIDs tested in BOTH (common):      {len(common_cids):,}")
print()
print(f"Caspase Active CIDs:               {len(caspase_active_cids):,}")
print(f"Caspase Inactive CIDs:             {len(caspase_inactive_cids):,}")
print(f"Caspase Truly Inactive (in common): {len(caspase_truly_inactive):,}")
print()
print(f"Viability Active CIDs:             {len(viability_active_cids):,}")
print(f"Viability Inactive CIDs:           {len(viability_inactive_cids):,}")
print(f"Viability Truly Inactive (in common): {len(viability_truly_inactive):,}")
print()

# Case 4: Caspase Truly Inactive AND Viability Truly Inactive
case4_cids = caspase_truly_inactive.intersection(viability_truly_inactive)

print(f"CASE 4: Caspase=Inactive AND Viability=Inactive: {len(case4_cids):,}")
print()
print("These are NON-TOXIC compounds at tested concentrations")

STRICT METHOD: Active takes precedence, Inactive = has Inactive & no Active
CIDs tested in Caspase assays:     7,716
CIDs tested in Viability assays:   8,734
CIDs tested in BOTH (common):      7,673

Caspase Active CIDs:               324
Caspase Inactive CIDs:             7,534
Caspase Truly Inactive (in common): 7,296

Viability Active CIDs:             1,273
Viability Inactive CIDs:           8,213
Viability Truly Inactive (in common): 6,367

CASE 4: Caspase=Inactive AND Viability=Inactive: 6,305

These are NON-TOXIC compounds at tested concentrations


In [4]:
# Show count of CIDs (list may be too long to display)
print(f"Total {len(case4_cids):,} CIDs with Caspase=Never Active AND Viability=Never Active")
print(f"\nFirst 100 CIDs:")
print(sorted(case4_cids)[:100])

Total 6,305 CIDs with Caspase=Never Active AND Viability=Never Active

First 100 CIDs:
[4, 11, 13, 33, 34, 51, 66, 72, 79, 107, 135, 174, 176, 178, 179, 180, 190, 196, 204, 227, 240, 241, 243, 244, 247, 261, 262, 263, 264, 300, 323, 325, 326, 332, 335, 338, 342, 356, 379, 385, 398, 447, 454, 460, 464, 525, 527, 541, 546, 564, 568, 594, 597, 612, 650, 660, 674, 679, 681, 700, 701, 702, 712, 713, 727, 743, 750, 753, 757, 774, 795, 802, 803, 811, 863, 864, 867, 875, 880, 887, 892, 896, 904, 931, 936, 938, 949, 957, 967, 970, 971, 978, 985, 991, 992, 995, 996, 997, 998, 999]


---
## Block 2: Case 4 Analysis by Cell Line (SAME CELL LINE)

For each cell line, find CIDs where Caspase=Inactive AND Viability=Inactive in the SAME cell line.

In [5]:
# Analysis by Cell Line - Caspase Inactive & Viability Inactive (SAME CELL LINE)
print("="*70)
print("CASE 4: CASPASE=INACTIVE AND VIABILITY=INACTIVE BY CELL LINE")
print("="*70)
print("Note: Viability data exists only for HepG2 and CHO-K1")
print()

# Get caspase and viability dataframes
caspase_df = df[df['assay_type'] == 'caspase']
viability_df = df[df['assay_type'].isin(VIABILITY_TYPES)]

caspase_cell_lines = sorted(caspase_df['cell_line'].unique())

results = []
for cell in caspase_cell_lines:
    # Total CIDs tested for caspase in this cell line
    caspase_in_cell = caspase_df[caspase_df['cell_line'] == cell]
    total_caspase_tested = caspase_in_cell['CID'].nunique()
    
    # Caspase Inactive in this cell line
    casp_inactive = set(caspase_in_cell[caspase_in_cell['outcome'] == 'Inactive']['CID'].unique())
    
    # Viability Inactive in THIS SAME cell line
    viability_in_cell = viability_df[viability_df['cell_line'] == cell]
    total_viability_tested = viability_in_cell['CID'].nunique()
    viab_inactive = set(viability_in_cell[viability_in_cell['outcome'] == 'Inactive']['CID'].unique())
    
    # Case 4: Caspase Inactive AND Viability Inactive in SAME cell line
    case4_in_cell = casp_inactive.intersection(viab_inactive)
    
    results.append({
        'Cell Line': cell,
        'Caspase Tested': total_caspase_tested,
        'Caspase Inactive': len(casp_inactive),
        'Viability Tested': total_viability_tested,
        'Viability Inactive': len(viab_inactive),
        'Case4 (Both)': len(case4_in_cell),
        'Case4 Rate (%)': round(len(case4_in_cell)/len(casp_inactive)*100, 2) if len(casp_inactive) > 0 else 0
    })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('Caspase Inactive', ascending=False)
print(results_df.to_string(index=False))

# Summary for cell lines WITH viability data
print(f"\n" + "-"*70)
hepg2_case4 = results_df[results_df['Cell Line'] == 'HepG2']['Case4 (Both)'].values[0]
chok1_case4 = results_df[results_df['Cell Line'] == 'CHO-K1']['Case4 (Both)'].values[0]
print(f"HepG2: {hepg2_case4} CIDs with Caspase=Inactive AND Viability=Inactive")
print(f"CHO-K1: {chok1_case4} CIDs with Caspase=Inactive AND Viability=Inactive")

CASE 4: CASPASE=INACTIVE AND VIABILITY=INACTIVE BY CELL LINE
Note: Viability data exists only for HepG2 and CHO-K1

            Cell Line  Caspase Tested  Caspase Inactive  Viability Tested  Viability Inactive  Case4 (Both)  Case4 Rate (%)
                HepG2            7671              7299              8734                8200          6938           95.05
               CHO-K1             394               199              7671                5743            12            6.03
             HUV-EC-C              27                22                 0                   0             0            0.00
               Jurkat             106                 4                 0                   0             0            0.00
               SHSY5Y              23                 3                 0                   0             0            0.00
                   BJ               5                 2                 0                   0             0            0.00
            Mesa

In [6]:
# HepG2 Specific: Case 4 IN THE SAME CELL LINE
print("="*70)
print("HepG2: CASPASE=INACTIVE AND VIABILITY=INACTIVE (SAME CELL LINE)")
print("="*70)

# Caspase Inactive in HepG2
caspase_inactive_hepg2 = set(df[
    (df['assay_type'] == 'caspase') & 
    (df['cell_line'] == 'HepG2') & 
    (df['outcome'] == 'Inactive')
]['CID'].unique())

# Viability Inactive in HepG2 (mitochondrial_viability_hepg2 + caspase_viability_hepg2)
viability_inactive_hepg2 = set(df[
    (df['assay_type'].isin(['mitochondrial_viability_hepg2', 'caspase_viability_hepg2'])) & 
    (df['cell_line'] == 'HepG2') & 
    (df['outcome'] == 'Inactive')
]['CID'].unique())

case4_hepg2 = caspase_inactive_hepg2.intersection(viability_inactive_hepg2)

print(f"\nCaspase Inactive in HepG2: {len(caspase_inactive_hepg2):,}")
print(f"Viability Inactive in HepG2 (mito + caspase_viability): {len(viability_inactive_hepg2):,}")
print(f"\n>>> CASE 4 in HepG2: {len(case4_hepg2):,}")

HepG2: CASPASE=INACTIVE AND VIABILITY=INACTIVE (SAME CELL LINE)

Caspase Inactive in HepG2: 7,299
Viability Inactive in HepG2 (mito + caspase_viability): 8,200

>>> CASE 4 in HepG2: 6,938


In [7]:
# CHO-K1 Specific: Case 4 IN THE SAME CELL LINE
print("="*70)
print("CHO-K1: CASPASE=INACTIVE AND VIABILITY=INACTIVE (SAME CELL LINE)")
print("="*70)

# Caspase Inactive in CHO-K1
caspase_inactive_chok1 = set(df[
    (df['assay_type'] == 'caspase') & 
    (df['cell_line'] == 'CHO-K1') & 
    (df['outcome'] == 'Inactive')
]['CID'].unique())

# Viability Inactive in CHO-K1
viability_inactive_chok1 = set(df[
    (df['assay_type'] == 'viability') & 
    (df['cell_line'] == 'CHO-K1') & 
    (df['outcome'] == 'Inactive')
]['CID'].unique())

case4_chok1 = caspase_inactive_chok1.intersection(viability_inactive_chok1)

print(f"\nCaspase Inactive in CHO-K1: {len(caspase_inactive_chok1):,}")
print(f"Viability Inactive in CHO-K1: {len(viability_inactive_chok1):,}")
print(f"\n>>> CASE 4 in CHO-K1: {len(case4_chok1):,}")

CHO-K1: CASPASE=INACTIVE AND VIABILITY=INACTIVE (SAME CELL LINE)

Caspase Inactive in CHO-K1: 199
Viability Inactive in CHO-K1: 5,743

>>> CASE 4 in CHO-K1: 12


---
## Block 3: Export Data - Case 4

Save the data to CSV for easy verification, grouped by CID.

In [8]:
# Case 4: Export CIDs where Caspase=Inactive (truly) AND Viability=Inactive (truly)
print("="*70)
print("CASE 4: EXPORTING CASPASE INACTIVE & VIABILITY INACTIVE DATA")
print("="*70)

print(f"\nTotal CIDs (strict method): {len(case4_cids):,}")

# Get records for these CIDs:
# - All Caspase records (to show outcomes - should have Inactive but no Active)
# - All Viability records (to show outcomes - should have Inactive but no Active)
case4_caspase = df[
    (df['CID'].isin(case4_cids)) & 
    (df['assay_type'] == 'caspase')
][['CID', 'SMILES', 'assay_type', 'cell_line', 'outcome', 'activity_score', 'AID', 'assay_name']].copy()

case4_viability = df[
    (df['CID'].isin(case4_cids)) & 
    (df['assay_type'].isin(VIABILITY_TYPES))
][['CID', 'SMILES', 'assay_type', 'cell_line', 'outcome', 'activity_score', 'AID', 'assay_name']].copy()

# Combine
case4_data = pd.concat([case4_caspase, case4_viability], ignore_index=True)
case4_data = case4_data.sort_values(['CID', 'cell_line', 'assay_type'])

print(f"Total records: {len(case4_data):,}")
print(f"\nPreview (first 20 rows):")
print(case4_data.head(20).to_string(index=False))

# Verify: Both should have Inactive but NO Active
print(f"\nCaspase outcomes for Case 4 CIDs (should have Inactive, NO Active):")
print(case4_caspase['outcome'].value_counts())
print(f"\nViability outcomes for Case 4 CIDs (should have Inactive, NO Active):")
print(case4_viability['outcome'].value_counts())

# Save to CSV
output_path = 'output/case4_caspase_inactive_viability_inactive.csv'
case4_data.to_csv(output_path, index=False)
print(f"\n>>> Saved to: {output_path}")
print(f"    Shape: {case4_data.shape}")

CASE 4: EXPORTING CASPASE INACTIVE & VIABILITY INACTIVE DATA

Total CIDs (strict method): 6,305
Total records: 19,951

Preview (first 20 rows):
 CID                SMILES                    assay_type cell_line  outcome  activity_score     AID                   assay_name
   4               CC(CN)O                     viability    CHO-K1 Inactive               0 1346979 AID_1346979_viability_CHO-K1
   4               CC(CN)O                       caspase     HepG2 Inactive               0 1346978  AID_1346978_caspase37_HepG2
   4               CC(CN)O mitochondrial_viability_hepg2     HepG2 Inactive               0  720634        AID_720634_mito_HepG2
  11              C(CCl)Cl                     viability    CHO-K1 Inactive               0 1346979 AID_1346979_viability_CHO-K1
  11              C(CCl)Cl                       caspase     HepG2 Inactive               0     654       AID_654_caspase3_HepG2
  11              C(CCl)Cl mitochondrial_viability_hepg2     HepG2 Inactive       

In [9]:
# Create summary table grouped by CID for easy count verification
print("="*70)
print("CASE 4: SUMMARY BY CID (STRICT METHOD)")
print("="*70)

# For each CID, get the outcomes for caspase and viability
caspase_by_cid = case4_caspase.groupby('CID').agg({
    'cell_line': lambda x: ', '.join(sorted(x.unique())),
    'outcome': lambda x: ', '.join(sorted(x.unique()))
}).reset_index()
caspase_by_cid.columns = ['CID', 'Caspase_CellLines', 'Caspase_Outcomes']

viability_by_cid = case4_viability.groupby('CID').agg({
    'cell_line': lambda x: ', '.join(sorted(x.unique())),
    'assay_type': lambda x: ', '.join(sorted(x.unique())),
    'outcome': lambda x: ', '.join(sorted(x.unique()))
}).reset_index()
viability_by_cid.columns = ['CID', 'Viability_CellLines', 'Viability_AssayTypes', 'Viability_Outcomes']

# Merge
case4_summary = pd.merge(caspase_by_cid, viability_by_cid, on='CID')

# Add SMILES
smiles_map = df[['CID', 'SMILES']].drop_duplicates().set_index('CID')['SMILES'].to_dict()
case4_summary['SMILES'] = case4_summary['CID'].map(smiles_map)

case4_summary = case4_summary.sort_values('CID')

print(f"\nTotal unique CIDs: {len(case4_summary):,}")
print(f"\nSummary table (first 30 rows):")
print(case4_summary.head(30).to_string(index=False))

# Save summary to CSV
summary_path = 'output/case4_summary_by_cid.csv'
case4_summary.to_csv(summary_path, index=False)
print(f"\n>>> Summary saved to: {summary_path}")
print(f"    Total CIDs: {len(case4_summary):,}")

CASE 4: SUMMARY BY CID (STRICT METHOD)

Total unique CIDs: 6,305

Summary table (first 30 rows):
 CID Caspase_CellLines Caspase_Outcomes Viability_CellLines                                              Viability_AssayTypes     Viability_Outcomes                   SMILES
   4             HepG2         Inactive       CHO-K1, HepG2                          mitochondrial_viability_hepg2, viability               Inactive                  CC(CN)O
  11             HepG2         Inactive       CHO-K1, HepG2                          mitochondrial_viability_hepg2, viability               Inactive                 C(CCl)Cl
  13             HepG2         Inactive       CHO-K1, HepG2                          mitochondrial_viability_hepg2, viability               Inactive    C1=CC(=C(C=C1Cl)Cl)Cl
  33             HepG2         Inactive       CHO-K1, HepG2                          mitochondrial_viability_hepg2, viability               Inactive                 C(C=O)Cl
  34             HepG2         In